# 🩺 Prédiction du Diabète — Régression Logistique
**Exercices 1 à 6** : Chargement · Standardisation · Entraînement · Évaluation · Frontière de décision · Courbe ROC

## 📦 Imports & configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    precision_score, recall_score, f1_score,
    roc_curve, roc_auc_score
)

# Style global dark
plt.rcParams.update({
    "figure.facecolor": "#0f1117",
    "axes.facecolor":   "#1a1d2e",
    "axes.edgecolor":   "#3a3d5c",
    "axes.labelcolor":  "#e0e0f0",
    "xtick.color":      "#b0b0d0",
    "ytick.color":      "#b0b0d0",
    "text.color":       "#e0e0f0",
    "grid.color":       "#2a2d45",
    "grid.alpha":       0.5,
    "font.family":      "DejaVu Sans",
})
PALETTE = ["#6c63ff", "#ff6584", "#43e97b", "#f7971e", "#4facfe"]
print("✅ Imports OK")

---
## 🌟 Exercice 1 — Compréhension du problème & collecte de données
> Charger le dataset, explorer les classes, diviser en train/test.

In [ ]:
# ── Chargement du dataset ────────────────────────────────────────────────────
url = "https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv"
df = pd.read_csv(url)

print(f"Dimensions : {df.shape}")
print(f"
Aperçu :
{df.head()}")
print(f"
Valeurs manquantes :
{df.isnull().sum()}")
print(f"
Statistiques :
{df.describe().round(2)}")

In [ ]:
# ── Distribution de la cible ─────────────────────────────────────────────────
target_col = "Outcome"
counts = df[target_col].value_counts()
print(f"Cas négatifs (0) : {counts[0]}  |  Cas positifs (1) : {counts[1]}")
print(f"Ratio positifs   : {counts[1]/len(df)*100:.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Exercice 1 — Exploration des données", fontsize=15, fontweight="bold")

# Bar chart classes
ax = axes[0]
bars = ax.bar(["Négatif (0)", "Positif (1)"], [counts[0], counts[1]],
             color=[PALETTE[0], PALETTE[1]], edgecolor="#0f1117", width=0.5)
for bar, val in zip(bars, [counts[0], counts[1]]):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+5,
            f"{val}
({val/len(df)*100:.1f}%)", ha="center", fontsize=11)
ax.set_title("Distribution de la variable cible")
ax.set_ylabel("Nombre d'individus")
ax.set_ylim(0, max(counts)*1.25)
ax.grid(axis="y")

# Heatmap corrélation
ax2 = axes[1]
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, ax=ax2, cmap="coolwarm", center=0,
            linewidths=0.5, annot=True, fmt=".2f",
            annot_kws={"size": 7}, cbar_kws={"shrink": 0.8})
ax2.set_title("Matrice de corrélation")
plt.tight_layout()
plt.show()

In [ ]:
# ── Séparation features / cible & split train/test ───────────────────────────
X = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train : {X_train.shape[0]} exemples  |  Test : {X_test.shape[0]} exemples")
print(f"Features : {list(X.columns)}")

---
## 🌟 Exercice 2 — Sélection du modèle & standardisation
> Quel modèle choisir ? Faut-il normaliser ?

In [ ]:
# ── Justification du choix de modèle ────────────────────────────────────────
print("""
Modèle choisi : Régression Logistique
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• Problème de classification binaire (diabète : oui/non).
• La régression logistique modélise P(Y=1|X) via une fonction sigmoïde.
• Avantages :
    - Interprétable : les coefficients indiquent l'importance de chaque feature.
    - Fournit des probabilités calibrées (utile en médecine).
    - Converge rapidement sur des datasets tabulaires de taille modérée.
    - Robuste face à des features peu corrélées entre elles.

Normalisation : OUI — StandardScaler
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• La régression logistique optimise par descente de gradient.
• Des features à échelles très différentes (ex: Glucose~120 vs BMI~32)
  ralentissent la convergence et biaisent les coefficients.
• StandardScaler : μ=0, σ=1 pour chaque feature.
• Important : fit sur train uniquement, transform sur train ET test.
""")

In [ ]:
# ── Application du StandardScaler ───────────────────────────────────────────
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)   # fit + transform sur le train
X_test_sc  = scaler.transform(X_test)        # transform seulement sur le test

# Visualisation avant / après standardisation
fig, axes = plt.subplots(2, 4, figsize=(18, 7))
fig.suptitle("Exercice 2 — Distributions avant / après StandardScaler",
             fontsize=14, fontweight="bold")

feature_names = X.columns.tolist()
X_train_df    = pd.DataFrame(X_train_sc, columns=feature_names)

for i, feat in enumerate(feature_names):
    ax = axes[i // 4][i % 4]
    ax.hist(X_train[feat], bins=25, alpha=0.55, color=PALETTE[0],
            label="Avant", edgecolor="none")
    ax.hist(X_train_df[feat], bins=25, alpha=0.55, color=PALETTE[1],
            label="Après", edgecolor="none")
    ax.set_title(feat, fontsize=9)
    ax.legend(fontsize=7)
    ax.grid(axis="y")

plt.tight_layout()
plt.show()
print("✅ StandardScaler appliqué")

---
## 🌟 Exercice 3 — Entraînement du modèle
> Entraîner la régression logistique et visualiser les coefficients.

In [ ]:
# ── Entraînement ─────────────────────────────────────────────────────────────
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_sc, y_train)

y_pred      = model.predict(X_test_sc)
y_pred_prob = model.predict_proba(X_test_sc)[:, 1]

print(f"Modèle entraîné ✅")
print(f"Intercept : {model.intercept_[0]:.4f}")
print(f"
Coefficients :")
for feat, coef in zip(feature_names, model.coef_[0]):
    print(f"  {feat:<30} {coef:+.4f}")

In [ ]:
# ── Visualisation des coefficients (importance des features) ─────────────────
coefs  = pd.Series(model.coef_[0], index=feature_names).sort_values()
colors = [PALETTE[1] if c < 0 else PALETTE[0] for c in coefs]

fig, ax = plt.subplots(figsize=(10, 5))
fig.suptitle("Exercice 3 — Coefficients de la régression logistique",
             fontsize=13, fontweight="bold")
bars = ax.barh(coefs.index, coefs.values, color=colors, edgecolor="#0f1117")
ax.axvline(0, color="white", linewidth=1.2, linestyle="--")
for bar, val in zip(bars, coefs.values):
    ax.text(val + (0.02 if val >= 0 else -0.02), bar.get_y()+bar.get_height()/2,
            f"{val:+.3f}", va="center",
            ha="left" if val >= 0 else "right", fontsize=9)
ax.set_xlabel("Coefficient (échelle standardisée)")
ax.set_title("Un coeff positif augmente la probabilité de diabète", fontsize=10)
ax.grid(axis="x")
patch_pos = mpatches.Patch(color=PALETTE[0], label="Facteur de risque (+)")
patch_neg = mpatches.Patch(color=PALETTE[1], label="Facteur protecteur (-)")
ax.legend(handles=[patch_pos, patch_neg])
plt.tight_layout()
plt.show()

---
## 🌟 Exercice 4 — Métriques d'évaluation

In [ ]:
# ── Calcul des métriques ─────────────────────────────────────────────────────
acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)
cm   = confusion_matrix(y_test, y_pred)

print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"F1-Score  : {f1:.4f}")
print(f"
{classification_report(y_test, y_pred)}")

In [ ]:
# ── Visualisation : 3 graphiques ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Exercice 4 — Métriques d'évaluation", fontsize=15, fontweight="bold")

# 4a — Accuracy train vs test
ax = axes[0]
accs = [accuracy_score(y_train, model.predict(X_train_sc)), acc]
bars = ax.bar(["Train", "Test"], accs,
             color=[PALETTE[0], PALETTE[2]], edgecolor="#0f1117", width=0.4)
for bar, val in zip(bars, accs):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
            f"{val:.3f}", ha="center", fontsize=12, fontweight="bold")
ax.set_ylim(0, 1.1)
ax.set_title("Score d'Accuracy")
ax.set_ylabel("Accuracy")
ax.axhline(0.5, color="#ff6584", linestyle="--", alpha=0.7, label="Baseline (0.5)")
ax.legend()
ax.grid(axis="y")

# 4b — Matrice de confusion
ax = axes[1]
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
           xticklabels=["Prédit 0","Prédit 1"],
           yticklabels=["Réel 0","Réel 1"],
           linewidths=2, linecolor="#0f1117",
           annot_kws={"size":14,"weight":"bold"})
ax.set_title("Matrice de Confusion")
ax.set_xlabel("Classe prédite")
ax.set_ylabel("Classe réelle")

# 4c — Precision / Recall / F1
ax = axes[2]
names  = ["Precision","Recall","F1-Score"]
vals   = [prec, rec, f1]
bars = ax.bar(names, vals, color=[PALETTE[0],PALETTE[1],PALETTE[3]],
             edgecolor="#0f1117", width=0.5)
for bar, val in zip(bars, vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
            f"{val:.3f}", ha="center", fontsize=12, fontweight="bold")
ax.set_ylim(0, 1.1)
ax.set_title("Precision, Recall & F1-Score")
ax.grid(axis="y")

plt.tight_layout()
plt.show()

print("""
Commentaires :
  Accuracy  : mesure globale — attention si classes déséquilibrées.
  Precision : parmi les individus prédits diabétiques, combien le sont vraiment ?
              → limiter les faux positifs (alarmes inutiles).
  Recall    : parmi les vrais diabétiques, combien le modèle détecte-t-il ?
              → critique en médecine : rater un cas coûte cher.
  F1-Score  : moyenne harmonique Precision/Recall — meilleur indicateur global.
""")

---
## 🌟 Exercice 5 — Frontière de décision (PCA 2D)

In [ ]:
# ── Réduction PCA → 2D pour visualisation ────────────────────────────────────
pca = PCA(n_components=2, random_state=42)
X_train_2d = pca.fit_transform(X_train_sc)
X_test_2d  = pca.transform(X_test_sc)

model_2d = LogisticRegression(max_iter=1000, random_state=42)
model_2d.fit(X_train_2d, y_train)
acc_2d = accuracy_score(y_test, model_2d.predict(X_test_2d))

# Grille de décision
x_min, x_max = X_test_2d[:,0].min()-1, X_test_2d[:,0].max()+1
y_min, y_max = X_test_2d[:,1].min()-1, X_test_2d[:,1].max()+1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 400),
                     np.linspace(y_min, y_max, 400))
Z = model_2d.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:,1].reshape(xx.shape)

fig, ax = plt.subplots(figsize=(10, 7))
fig.patch.set_facecolor("#0f1117")

cf = ax.contourf(xx, yy, Z, levels=50, cmap="RdBu_r", alpha=0.75)
ax.contour(xx, yy, Z, levels=[0.5], colors="white", linewidths=2, linestyles="--")
plt.colorbar(cf, ax=ax, label="Probabilité diabète (classe 1)")

colors_pt = np.where(y_test==1, PALETTE[1], PALETTE[0])
ax.scatter(X_test_2d[:,0], X_test_2d[:,1], c=colors_pt,
           edgecolors="white", linewidths=0.4, s=40, alpha=0.85, zorder=3)

p0 = mpatches.Patch(color=PALETTE[0], label="Non-diabétique (0)")
p1 = mpatches.Patch(color=PALETTE[1], label="Diabétique (1)")
ax.legend(handles=[p0,p1], loc="upper right", framealpha=0.3)
ax.set_title(f"Frontière de décision (PCA 2D) — Accuracy : {acc_2d:.3f}",
             fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var.)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var.)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("Note : la réduction PCA à 2D fait perdre de l'information →",
      f"accuracy 2D={acc_2d:.3f} vs modèle complet={acc:.3f}")

---
## 🌟 Exercice 6 — Courbe ROC

In [ ]:
# ── Calcul & tracé de la courbe ROC ─────────────────────────────────────────
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)
auc = roc_auc_score(y_test, y_pred_prob)
print(f"AUC-ROC : {auc:.4f}")

fig, ax = plt.subplots(figsize=(8, 6))
fig.patch.set_facecolor("#0f1117")

ax.plot(fpr, tpr, color=PALETTE[0], lw=2.5,
        label=f"ROC Curve (AUC = {auc:.3f})")
ax.fill_between(fpr, tpr, alpha=0.15, color=PALETTE[0])
ax.plot([0,1],[0,1], color="#ff6584", lw=1.5, linestyle="--",
        label="Aléatoire (AUC = 0.5)")

# Seuil optimal (Youden)
best_idx = np.argmax(tpr - fpr)
ax.scatter(fpr[best_idx], tpr[best_idx], s=120, zorder=5,
          color=PALETTE[2], edgecolors="white",
          label=f"Seuil optimal = {thresholds[best_idx]:.2f}")

ax.set_xlim([-0.01, 1.01])
ax.set_ylim([-0.01, 1.05])
ax.set_xlabel("Taux de Faux Positifs (FPR)", fontsize=12)
ax.set_ylabel("Taux de Vrais Positifs (TPR / Recall)", fontsize=12)
ax.set_title("Courbe ROC — Régression Logistique",
             fontsize=13, fontweight="bold", pad=12)
ax.legend(loc="lower right", framealpha=0.3)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

print("""
Interprétation :
  AUC = 1.0 → modèle parfait.
  AUC = 0.5 → modèle aléatoire (diagonale).
  Plus la courbe est proche du coin supérieur gauche, meilleur est le modèle.
  Le seuil optimal (Youden) maximise TPR − FPR :
  bon équilibre entre détecter les vrais cas et limiter les fausses alarmes.
""")